# Solar Irradiance Prediction — Full Pipeline Notebook
**Mustang, Nepal (28.94°N, 83.90°E) · NASA POWER daily data · 1 Jan 1995 to 1 Jan 2026**

This single notebook combines: data cleaning → EDA → five ML models → comparison.

| Section | Algorithm |
|---|---|
| 1 | Data Cleaning & Filtering |
| 2 | Exploratory Data Analysis (EDA) |
| 3 | Multiple Linear Regression |
| 4 | Decision Tree Regressor |
| 5 | Random Forest Regressor |
| 6 | Gradient Boosting Regressor |
| 7 | K-Nearest Neighbours (KNN) Regressor |
| 8 | Model Comparison Summary |

## 0. Shared Imports

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
print('All libraries loaded.')

---
## Section 1: Data Cleaning & Filtering

In [ ]:
import pandas as pd
import numpy as np


filename = 'POWER_Point_Daily_19950101_20260101_028d94N_083d90E_LST.csv'
df = pd.read_csv(filename, skiprows=23)

print(f"Raw data loaded. Initial Shape: {df.shape}")

df.replace(-999.0, np.nan, inplace=True)

median_kt = df['ALLSKY_KT'].median()
df['ALLSKY_KT'] = df['ALLSKY_KT'].fillna(median_kt)

print(f"Missing values handled. Remaining nulls in dataset: {df.isnull().sum().sum()}")

columns_to_drop = ['YEAR', 'MO', 'DY']
df_clean = df.drop(columns=columns_to_drop)

print(f"Data filtered. Final Shape ready for EDA and ML: {df_clean.shape}")

df_clean.to_csv('Cleaned_Mustang_Solar_Data.csv', index=False)
print("Saved as 'Cleaned_Mustang_Solar_Data.csv'.")

### Shared data setup (run once — used by all models below)

In [ ]:
# ── Shared: load cleaned data, define X/y, split ──────────────────
df = pd.read_csv('Cleaned_Mustang_Solar_Data.csv')

TARGET   = 'ALLSKY_SFC_SW_DWN'
LEAKY    = ['ALLSKY_KT']           # excluded: computed from the target

y = df[TARGET]
X = df.drop(columns=[TARGET] + LEAKY)

# Chronological 80/20 split
split_index = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

# Chronological validation split inside the training set (used for tuning)
val_index   = int(len(X_train) * 0.8)
X_subtrain  = X_train.iloc[:val_index];   X_val = X_train.iloc[val_index:]
y_subtrain  = y_train.iloc[:val_index];   y_val = y_train.iloc[val_index:]

print(f'Full dataset : {df.shape}')
print(f'Train        : {X_train.shape}  |  Test  : {X_test.shape}')
print(f'Sub-train    : {X_subtrain.shape}  |  Val   : {X_val.shape}')
print(f'Features ({len(X.columns)}): {list(X.columns)}')

---
## Section 2: Exploratory Data Analysis (EDA)

# Solar Irradiance EDA
Exploratory analysis of daily solar irradiance (`ALLSKY_SFC_SW_DWN`).

**Note:** this notebook expects `df` (raw data) and `df_clean` (cleaned numeric data) to already exist. Use the load cell below if you need to create them.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

sns.set_theme(style="whitegrid")

## 0. Load data (edit or skip)

## Descriptive statistics

In [ ]:
# Summary statistics for all numeric features
desc = df_clean.describe().T
desc['skew'] = df_clean.skew()
desc['kurtosis'] = df_clean.kurtosis()
desc['missing'] = df_clean.isnull().sum()
display(desc.round(3))

In [ ]:
# Target variable by month
monthly_stats = df.groupby('MO')['ALLSKY_SFC_SW_DWN'].agg(
    ['count', 'mean', 'median', 'std', 'min', 'max']
).round(3)
display(monthly_stats)

In [ ]:
# Target variable by year
yearly_stats = df.groupby('YEAR')['ALLSKY_SFC_SW_DWN'].agg(
    ['count', 'mean', 'median', 'std', 'min', 'max']
).round(3)
display(yearly_stats)

## 1. Feature correlation heatmap

In [ ]:
plt.figure(figsize=(14, 12))
sns.heatmap(df_clean.corr(), annot=True, fmt='.2f', annot_kws={'size': 8},
            cmap='coolwarm', linewidths=0.5, vmin=-1, vmax=1, center=0)
plt.title('1. Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Distribution of daily solar irradiance

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df_clean['ALLSKY_SFC_SW_DWN'], bins=50, kde=True, color='darkorange')
plt.title('2. Distribution of Daily Solar Irradiance', fontsize=14, fontweight='bold')
plt.xlabel('Solar Irradiance (kW-hr/m^2/day)', fontsize=12)
plt.ylabel('Frequency (Days)', fontsize=12)
plt.tight_layout()
plt.show()

## 3. Monthly seasonal trends

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(data=df, x='MO', y='ALLSKY_SFC_SW_DWN', hue='MO',
            palette='YlOrRd', legend=False)
plt.title('3. Monthly Seasonal Trends (1995-2026)', fontsize=14, fontweight='bold')
plt.xlabel('Month (1 = Jan, 12 = Dec)', fontsize=12)
plt.ylabel('Solar Irradiance', fontsize=12)
plt.tight_layout()
plt.show()

## 4. Ideal vs actual irradiance

In [ ]:
plt.figure(figsize=(10, 6))

if 'CLRSKY_SFC_SW_DWN' in df_clean.columns:
    sns.scatterplot(data=df_clean, x='CLRSKY_SFC_SW_DWN', y='ALLSKY_SFC_SW_DWN',
                    alpha=0.3, color='blue')
    plt.xlabel('Clear Sky Irradiance (Ideal Condition)', fontsize=12)
else:
    first_feat = df_clean.columns[0]
    sns.scatterplot(data=df_clean, x=first_feat, y='ALLSKY_SFC_SW_DWN',
                    alpha=0.3, color='blue')
    plt.xlabel(first_feat, fontsize=12)

plt.title('4. Feature Relationship: Ideal vs Actual Irradiance', fontsize=14, fontweight='bold')
plt.ylabel('Actual Solar Irradiance (Target)', fontsize=12)
plt.tight_layout()
plt.show()

## 5. Outlier detection

In [ ]:
plt.figure(figsize=(14, 6))
features_to_plot = df_clean.drop(columns=['ALLSKY_SFC_SW_DWN']).columns[:6]
sns.boxplot(data=df_clean[features_to_plot], palette='Set2')
plt.title('5. Outlier Detection in Meteorological Features', fontsize=14, fontweight='bold')
plt.xticks(rotation=45)
plt.ylabel('Feature Values', fontsize=12)
plt.tight_layout()
plt.show()

## 6. Yearly trend

In [ ]:
plt.figure(figsize=(14, 6))
yearly_data = df.groupby('YEAR')['ALLSKY_SFC_SW_DWN'].mean().reset_index()

sns.lineplot(data=yearly_data, x='YEAR', y='ALLSKY_SFC_SW_DWN',
             marker='o', color='purple', linewidth=2.5)
plt.title('6. Macro-Climate Trend: Yearly Average Solar Generation', fontsize=14, fontweight='bold')
plt.xlabel('Year', fontsize=12)
plt.ylabel('Avg Solar Irradiance', fontsize=12)
plt.xticks(yearly_data['YEAR'], rotation=45)
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

---
## Section 3: Multiple Linear Regression

In [ ]:
# Check for missing values
print("Missing values in X:", X_train.isnull().sum().sum())
print("Missing values in y:", y_train.isnull().sum())

# Check for non-numeric columns
print("Non-numeric columns:", X_train.select_dtypes(exclude=np.number).columns.tolist())

In [ ]:
# Create the Multiple Linear Regression model
mlr_model = LinearRegression()

# Train the model
mlr_model.fit(X_train, y_train)

print("Multiple Linear Regression model trained successfully!")

In [ ]:
# Predict solar irradiance
mlr_predictions = mlr_model.predict(X_test)

# Display the first 10 predictions
print("First 10 Predicted Values:")
print(mlr_predictions[:10])

In [ ]:
# Calculate evaluation metrics
mlr_r2 = r2_score(y_test, mlr_predictions)
mlr_mae = mean_absolute_error(y_test, mlr_predictions)
mlr_mse = mean_squared_error(y_test, mlr_predictions)
mlr_rmse = np.sqrt(mlr_mse)

# Display results
print("Multiple Linear Regression Model Evaluation")
print("--------------------------------------------")
print("R² Score:", r2)
print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)

In [ ]:
# Create a comparison DataFrame
comparison_df = pd.DataFrame({
    'Actual Solar Irradiance': y_test.values,
    'Predicted Solar Irradiance': mlr_predictions
})

# Display the first 10 results
print(comparison_df.head(10))

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(y_test, mlr_predictions, alpha=0.5)

# Reference line
plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    'r--'
)

plt.xlabel('Actual Solar Irradiance')
plt.ylabel('Predicted Solar Irradiance')
plt.title('Actual vs Predicted Solar Irradiance - MLR')
plt.grid(True)
plt.show()

In [ ]:
# Create a coefficient DataFrame
coefficients_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': mlr_model.coef_
})

# Display coefficients
print("Regression Coefficients:")
print(coefficients_df)

print("\nIntercept:", mlr_model.intercept_)

In [ ]:
# Save the trained MLR model
joblib.dump(mlr_model, 'multiple_linear_regression_model.pkl')

print("Multiple Linear Regression model saved successfully!")

---
## Section 4: Decision Tree Regressor

In [ ]:
with open('POWER_Point_Daily_19950101_20260101_028d94N_083d90E_LST.csv', 'r') as file:
    for i in range(30):
        print(i, file.readline().strip())

In [ ]:
print("Dataset 1:")
display(df1.head())

print("Dataset 2:")
display(df2.head())

print("Dataset 3:")
display(df3.head())

In [ ]:
print("Dataset 1 Shape:", df1.shape)
print("Dataset 2 Shape:", df2.shape)
print("Dataset 3 Shape:", df3.shape)

In [ ]:
print("Dataset 1 Columns:")
print(df1.columns.tolist())

print("\nDataset 2 Columns:")
print(df2.columns.tolist())

print("\nDataset 3 Columns:")
print(df3.columns.tolist())

In [ ]:
print("Missing Values in Dataset 1:")
print(df1.isnull().sum())

print("\nMissing Values in Dataset 2:")
print(df2.isnull().sum())

print("\nMissing Values in Dataset 3:")
print(df3.isnull().sum())

In [ ]:
print("Dataset 1 (-999 values):")
print((df1 == -999).sum())

print("\nDataset 2 (-999 values):")
print((df2 == -999).sum())

print("\nDataset 3 (-999 values):")
print((df3 == -999).sum())

In [ ]:
print("Dataset 1 Duplicates:", df1.duplicated().sum())

print("Dataset 2 Duplicates:", df2.duplicated().sum())

print("Dataset 3 Duplicates:", df3.duplicated().sum())

In [ ]:
print("Dataset 1 Information:")
df1.info()

print("\nDataset 2 Information:")
df2.info()

print("\nDataset 3 Information:")
df3.info()

In [ ]:
df = df1.copy()

print("Final Dataset Shape:", df.shape)
display(df.head())

In [ ]:
# Create Decision Tree model
model = DecisionTreeRegressor(
    max_depth=5,
    random_state=42
)

# Train the model
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

print("Decision Tree model trained successfully!")

In [ ]:
print("Actual Values:", y_test.head().values)
print("Predicted Values:", y_pred[:5])

In [ ]:
print("Initial Decision Tree Model (Max Depth = 5)")

print("R2 Score:", dtr_r2)

print("MAE:", dtr_mae)

print("MSE:", dtr_mse)

print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(y_test, y_pred, alpha=0.5)

plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    'r--',
    label='Perfect Prediction'
)

plt.xlabel("Actual Solar Irradiance")
plt.ylabel("Predicted Solar Irradiance")
plt.title("Actual vs Predicted Values - Decision Tree")
plt.legend()
plt.show()

In [ ]:
feature_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False)

display(feature_importance)

In [ ]:
importance = pd.Series(
    model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=True)

plt.figure(figsize=(9, 6))

importance.plot(kind='barh')

plt.xlabel("Feature Importance")
plt.ylabel("Features")
plt.title("Decision Tree Feature Importance")

plt.show()

In [ ]:
# Save the final model
joblib.dump(model, 'final_decision_tree_model.pkl')

print("Final Decision Tree model saved successfully!")


In [ ]:
depths = [3, 5, 7, 10, 12, 15, 20]
results = []

for depth in depths:
    dt_model = DecisionTreeRegressor(
        max_depth=depth,
        random_state=42
    )

    dt_model.fit(X_subtrain, y_subtrain)

    predictions = dt_model.predict(X_val)

    dtr_r2 = r2_score(y_val, predictions)
    dtr_mae = mean_absolute_error(y_val, predictions)
    rmse = np.sqrt(mean_squared_error(y_val, predictions))

    results.append({
        'Max Depth': depth,
        'R2 Score': r2,
        'MAE': mae,
        'RMSE': rmse
    })

tuning_results = pd.DataFrame(results)
display(tuning_results)

In [ ]:
best_depth = tuning_results.loc[
    tuning_results['R2 Score'].idxmax(),
    'Max Depth'
]

print("Best Max Depth:", best_depth)

In [ ]:
best_model = DecisionTreeRegressor(
    max_depth=int(best_depth),
    random_state=42
)

best_model.fit(X_train, y_train)

best_predictions = best_model.predict(X_test)

In [ ]:
dtr_r2 = r2_score(y_test, best_predictions)
dtr_mae = mean_absolute_error(y_test, best_predictions)
dtr_mse = mean_squared_error(y_test, best_predictions)
dtr_rmse = np.sqrt(dtr_mse)

print("Final Decision Tree Model Performance")
print("--------------------------------------")
print("Best Max Depth:", best_depth)
print("R2 Score:", r2)
print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)

In [ ]:
# Save the final Decision Tree model

joblib.dump(best_model, 'final_decision_tree_model.pkl')

print("Final Decision Tree Model saved successfully!")

In [ ]:
# Compare actual and predicted values

comparison = pd.DataFrame({
    'Actual Solar Irradiance': y_test.values,
    'Predicted Solar Irradiance': best_predictions
})

display(comparison.head(10))

In [ ]:
# Plot Actual vs Predicted Solar Irradiance

plt.figure(figsize=(8, 6))

plt.scatter(
    y_test,
    best_predictions,
    alpha=0.5
)

plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    'r--',
    label='Perfect Prediction'
)

plt.xlabel("Actual Solar Irradiance")
plt.ylabel("Predicted Solar Irradiance")
plt.title("Actual vs Predicted Solar Irradiance - Final Decision Tree")

plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Calculate Feature Importance

importance = pd.Series(
    best_model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

# Display Feature Importance
importance_df = pd.DataFrame({
    'Feature': importance.index,
    'Importance': importance.values
})

display(importance_df)

In [ ]:
# Plot Feature Importance

importance = pd.Series(
    best_model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=True)

plt.figure(figsize=(9, 6))

importance.plot(kind='barh')

plt.xlabel("Feature Importance")
plt.ylabel("Features")
plt.title("Feature Importance - Final Decision Tree")

plt.savefig(
    "decision_tree_feature_importance.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# Visualize the Final Decision Tree

plt.figure(figsize=(24, 12))

plot_tree(
    best_model,
    feature_names=X_train.columns,
    filled=True,
    rounded=True,
    fontsize=8
)

plt.title("Decision Tree Structure - Solar Irradiance Prediction")

plt.savefig(
    "decision_tree_structure.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# Display Final Model Information

print("Final Decision Tree Model Information")
print("--------------------------------------")

print("Algorithm: Decision Tree Regressor")
print("Best Max Depth:", best_depth)
print("Actual Tree Depth:", best_model.get_depth())
print("Number of Leaves:", best_model.get_n_leaves())
print("Number of Features:", X_train.shape[1])
print("Training Samples:", X_train.shape[0])
print("Testing Samples:", X_test.shape[0])

In [ ]:
# Create final evaluation results

final_results = pd.DataFrame({
    'Metric': ['R2 Score', 'MAE', 'MSE', 'RMSE'],
    'Value': [
        r2_score(y_test, best_predictions),
        mean_absolute_error(y_test, best_predictions),
        mean_squared_error(y_test, best_predictions),
        np.sqrt(mean_squared_error(y_test, best_predictions))
    ]
})

display(final_results)

# Save evaluation results
final_results.to_csv(
    'decision_tree_evaluation_results.csv',
    index=False
)

print("Evaluation results saved successfully!")

In [ ]:
import os

files = [
    'initial_decision_tree_model.pkl',
    'final_decision_tree_model.pkl',
    'decision_tree_structure.png',
    'decision_tree_feature_importance.png',
    'decision_tree_evaluation_results.csv'
]

print("Saved Files Verification")
print("------------------------")

for filename in files:
    if os.path.exists(filename):
        print(filename, ": Saved")
    else:
        print(filename, ": Not found")

---
## Section 5: Random Forest Regressor

In [ ]:
# Random Forest Regressor

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from sklearn.tree import plot_tree

from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

In [ ]:
# Create Random Forest Regressor

rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

# Train the model
rf_model.fit(X_train, y_train)

# Make predictions
rf_predictions = rf_model.predict(X_test)

print("Random Forest model trained successfully!")

In [ ]:
# Evaluate Random Forest Model

print("Initial Random Forest Model Performance")

print("R2 Score:", r2_score(y_test, rf_predictions))

print("MAE:", mean_absolute_error(y_test, rf_predictions))

print("MSE:", mean_squared_error(y_test, rf_predictions))

print("RMSE:", np.sqrt(mean_squared_error(y_test, rf_predictions)))

In [ ]:
comparison = pd.DataFrame({
    'Actual Solar Irradiance': y_test.values,
    'Predicted Solar Irradiance': rf_predictions
})

display(comparison.head(10))

In [ ]:
# Random Forest Model Evaluation

print("Initial Random Forest Model Performance")
print("R2 Score:", r2_score(y_test, rf_predictions))
print("MAE:", mean_absolute_error(y_test, rf_predictions))
print("MSE:", mean_squared_error(y_test, rf_predictions))
print("RMSE:", np.sqrt(mean_squared_error(y_test, rf_predictions)))

In [ ]:
# Random Forest Hyperparameter Tuning

n_estimators_values = [50, 100, 200]
max_depth_values = [5, 10, 15]

results = []

for n in n_estimators_values:
    for depth in max_depth_values:

        rf = RandomForestRegressor(
            n_estimators=n,
            max_depth=depth,
            random_state=42,
            n_jobs=-1
        )

        rf.fit(X_subtrain, y_subtrain)

        predictions = rf.predict(X_val)

        results.append({
            'N Estimators': n,
            'Max Depth': depth,
            'R2 Score': r2_score(y_val, predictions),
            'MAE': mean_absolute_error(y_val, predictions),
            'RMSE': np.sqrt(mean_squared_error(y_val, predictions))
        })

rf_tuning_results = pd.DataFrame(results)

display(rf_tuning_results)

In [ ]:
# Find the best Random Forest parameters

best_index = rf_tuning_results['R2 Score'].idxmax()

best_n_estimators = int(
    rf_tuning_results.loc[best_index, 'N Estimators']
)

best_max_depth = int(
    rf_tuning_results.loc[best_index, 'Max Depth']
)

print("Best Number of Trees:", best_n_estimators)
print("Best Maximum Depth:", best_max_depth)

In [ ]:
# Create the final Random Forest model

best_rf_model = RandomForestRegressor(
    n_estimators=best_n_estimators,
    max_depth=best_max_depth,
    random_state=42,
    n_jobs=-1
)

# Train using the complete training dataset
best_rf_model.fit(X_train, y_train)

# Predict using the test dataset
best_rf_predictions = best_rf_model.predict(X_test)

print("Final Random Forest model trained successfully!")

In [ ]:
# Final Random Forest Model Evaluation

rf_r2 = r2_score(y_test, best_rf_predictions)
rf_mae = mean_absolute_error(y_test, best_rf_predictions)
rf_mse = mean_squared_error(y_test, best_rf_predictions)
rf_rmse = np.sqrt(rf_mse)

print("Final Random Forest Model Performance")
print("-------------------------------------")
print("Best Number of Trees:", best_n_estimators)
print("Best Max Depth:", best_max_depth)
print("R2 Score:", rf_r2)
print("MAE:", rf_mae)
print("MSE:", rf_mse)
print("RMSE:", rf_rmse)

In [ ]:
# Compare Actual and Predicted Solar Irradiance

rf_comparison = pd.DataFrame({
    'Actual Solar Irradiance': y_test.values,
    'Predicted Solar Irradiance': best_rf_predictions
})

display(rf_comparison.head(10))

In [ ]:
# Plot Actual vs Predicted Solar Irradiance

plt.figure(figsize=(8, 6))

plt.scatter(
    y_test,
    best_rf_predictions,
    alpha=0.5,
    color='red'
)

plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    'r--',
    label='Perfect Prediction'
)

plt.xlabel("Actual Solar Irradiance")
plt.ylabel("Predicted Solar Irradiance")

plt.title("Actual vs Predicted Solar Irradiance - Random Forest")

plt.legend()
plt.grid(alpha=0.3)

plt.savefig(
    "random_forest_actual_vs_predicted.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# Calculate Random Forest Feature Importance

rf_importance = pd.Series(
    best_rf_model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

# Create Feature Importance Table

rf_importance_df = pd.DataFrame({
    'Feature': rf_importance.index,
    'Importance': rf_importance.values
})

display(rf_importance_df)

In [ ]:
# Plot Random Forest Feature Importance

plt.figure(figsize=(9, 6))

rf_importance.sort_values(ascending=True).plot(
    kind='barh',
    color='red'
)

plt.xlabel("Feature Importance")
plt.ylabel("Features")

plt.title("Feature Importance - Random Forest")

plt.grid(axis='x', alpha=0.3)

plt.savefig(
    "random_forest_feature_importance.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# Create Final Random Forest Evaluation Results

rf_final_results = pd.DataFrame({
    'Metric': ['R2 Score', 'MAE', 'MSE', 'RMSE'],
    'Value': [
        rf_r2,
        rf_mae,
        rf_mse,
        rf_rmse
    ]
})

display(rf_final_results)

# Save results
rf_final_results.to_csv(
    'random_forest_evaluation_results.csv',
    index=False
)

print("Random Forest evaluation results saved successfully!")

In [ ]:
import os

files = [
    'final_random_forest_model.pkl',
    'random_forest_actual_vs_predicted.png',
    'random_forest_feature_importance.png',
    'random_forest_evaluation_results.csv'
]

print("Random Forest Saved Files Verification")
print("--------------------------------------")

for filename in files:
    print(
        filename,
        ":",
        "Saved" if os.path.exists(filename) else "Not found"
    )

In [ ]:
# Save the final Random Forest model

joblib.dump(
    best_rf_model,
    'final_random_forest_model.pkl'
)

print("Random Forest model saved successfully!")

In [ ]:
import os

print("File exists:", os.path.exists('final_random_forest_model.pkl'))
print("File location:", os.path.abspath('final_random_forest_model.pkl'))

In [ ]:
# Final Random Forest Decision Tree Visualization

plt.figure(figsize=(20, 10))

plot_tree(
    best_rf_model.estimators_[0],
    feature_names=X_train.columns,
    filled=True,
    rounded=True,
    max_depth=3,
    fontsize=8
)

plt.title("Final Random Forest Decision Tree Visualization")
plt.show()

In [ ]:
# Final Random Forest Decision Tree Visualization

from sklearn.tree import plot_tree

plt.figure(figsize=(20, 10))

plot_tree(
    best_rf_model.estimators_[0],
    feature_names=X_train.columns,
    filled=True,
    rounded=True,
    max_depth=3,
    fontsize=8
)

plt.title("Final Random Forest Decision Tree Visualization")
plt.show()

---
## Section 6: Gradient Boosting Regressor

In [ ]:
# Hyperparameter candidates
n_estimators_values = [50, 100, 150]
learning_rate_values = [0.03, 0.05, 0.1]
max_depth_values = [2, 3, 4]

results = []

for n in n_estimators_values:
    for lr in learning_rate_values:
        for depth in max_depth_values:

            gb_model = GradientBoostingRegressor(
                n_estimators=n,
                learning_rate=lr,
                max_depth=depth,
                random_state=42
            )

            gb_model.fit(X_subtrain, y_subtrain)

            validation_predictions = gb_model.predict(X_validation)

            r2 = r2_score(y_validation, validation_predictions)
            mae = mean_absolute_error(y_validation, validation_predictions)
            mse = mean_squared_error(y_validation, validation_predictions)
            rmse = np.sqrt(mse)

            results.append({
                'n_estimators': n,
                'learning_rate': lr,
                'max_depth': depth,
                'R2': r2,
                'MAE': mae,
                'MSE': mse,
                'RMSE': rmse
            })

gb_tuning_results = pd.DataFrame(results)

display(
    gb_tuning_results.sort_values(
        by='RMSE'
    ).reset_index(drop=True)
)

In [ ]:
best_result = gb_tuning_results.loc[
    gb_tuning_results['RMSE'].idxmin()
]

best_n_estimators = int(best_result['n_estimators'])
best_learning_rate = float(best_result['learning_rate'])
best_max_depth = int(best_result['max_depth'])

print("Best Hyperparameters:")
print("Number of Estimators:", best_n_estimators)
print("Learning Rate:", best_learning_rate)
print("Maximum Depth:", best_max_depth)
print("Best Validation RMSE:", best_result['RMSE'])

In [ ]:
# Create final Gradient Boosting model
gb_model = GradientBoostingRegressor(
    n_estimators=best_n_estimators,
    learning_rate=best_learning_rate,
    max_depth=best_max_depth,
    random_state=42
)

# Train the model
gb_model.fit(X_train, y_train)

print("Gradient Boosting Regression model trained successfully!")

In [ ]:
gb_predictions = gb_model.predict(X_test)

print("Gradient Boosting predictions generated successfully!")

In [ ]:
gb_r2 = r2_score(y_test, gb_predictions)
gb_mae = mean_absolute_error(y_test, gb_predictions)
gb_mse = mean_squared_error(y_test, gb_predictions)
gb_rmse = np.sqrt(gb_mse)

gb_results = pd.DataFrame({
    'Metric': ['R2 Score', 'MAE', 'MSE', 'RMSE'],
    'Value': [gb_r2, gb_mae, gb_mse, gb_rmse]
})

display(gb_results)

In [ ]:
gb_comparison = pd.DataFrame({
    'Actual Values': y_test.values,
    'Predicted Values': gb_predictions
})

display(gb_comparison.head(10))

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(y_test, gb_predictions, alpha=0.5)

plt.xlabel('Actual Solar Irradiance')
plt.ylabel('Predicted Solar Irradiance')
plt.title('Gradient Boosting: Actual vs Predicted Values')

plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    'r--'
)

plt.show()

In [ ]:
joblib.dump(
    gb_model,
    'gradient_boosting_regression_model.pkl'
)

print("Gradient Boosting model saved successfully!")

---
## Section 7: K-Nearest Neighbours (KNN) Regressor

In [ ]:
scaler = StandardScaler()

X_subtrain_scaled = scaler.fit_transform(X_subtrain)
X_validation_scaled = scaler.transform(X_validation)

print("Sub-training data scaled:", X_subtrain_scaled.shape)
print("Validation data scaled:", X_validation_scaled.shape)

In [ ]:
# Hyperparameter candidates
k_values = [3, 5, 7, 9, 11, 15]
weight_values = ['uniform', 'distance']
p_values = [1, 2]

results = []

for k in k_values:
    for weight in weight_values:
        for p in p_values:

            knn = KNeighborsRegressor(
                n_neighbors=k,
                weights=weight,
                p=p
            )

            knn.fit(X_subtrain_scaled, y_subtrain)

            validation_predictions = knn.predict(X_validation_scaled)

            r2 = r2_score(y_validation, validation_predictions)
            mae = mean_absolute_error(y_validation, validation_predictions)
            mse = mean_squared_error(y_validation, validation_predictions)
            rmse = np.sqrt(mse)

            results.append({
                'n_neighbors': k,
                'weights': weight,
                'p': p,
                'R2': r2,
                'MAE': mae,
                'MSE': mse,
                'RMSE': rmse
            })

tuning_results = pd.DataFrame(results)

display(tuning_results.sort_values(by='RMSE').reset_index(drop=True))

In [ ]:
best_result = tuning_results.loc[tuning_results['RMSE'].idxmin()]

best_k = int(best_result['n_neighbors'])
best_weights = best_result['weights']
best_p = int(best_result['p'])

print("Best Hyperparameters:")
print("Number of Neighbors:", best_k)
print("Weights:", best_weights)
print("Distance Parameter (p):", best_p)
print("Best Validation RMSE:", best_result['RMSE'])

In [ ]:
# Scale the complete training and testing datasets
final_scaler = StandardScaler()

X_train_scaled = final_scaler.fit_transform(X_train)
X_test_scaled = final_scaler.transform(X_test)

# Create final KNN model
knn_model = KNeighborsRegressor(
    n_neighbors=best_k,
    weights=best_weights,
    p=best_p
)

# Train the model
knn_model.fit(X_train_scaled, y_train)

print("KNN Regression model trained successfully!")

In [ ]:
knn_predictions = knn_model.predict(X_test_scaled)

print("KNN predictions generated successfully!")

In [ ]:
knn_r2 = r2_score(y_test, knn_predictions)
knn_mae = mean_absolute_error(y_test, knn_predictions)
knn_mse = mean_squared_error(y_test, knn_predictions)
knn_rmse = np.sqrt(knn_mse)

knn_results = pd.DataFrame({
    'Metric': ['R2 Score', 'MAE', 'MSE', 'RMSE'],
    'Value': [knn_r2, knn_mae, knn_mse, knn_rmse]
})

display(knn_results)

In [ ]:
comparison = pd.DataFrame({
    'Actual Values': y_test.values,
    'Predicted Values': knn_predictions
})

display(comparison.head(10))

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(y_test, knn_predictions, alpha=0.5)

plt.xlabel('Actual Solar Irradiance')
plt.ylabel('Predicted Solar Irradiance')
plt.title('KNN Regression: Actual vs Predicted Values')

plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    'r--'
)

plt.show()

In [ ]:
joblib.dump(knn_model, 'knn_regression_model.pkl')
joblib.dump(final_scaler, 'knn_scaler.pkl')

print("KNN model and scaler saved successfully!")

---
## Section 8: Model Comparison Summary

In [ ]:
# ── Model Comparison Summary ────────────────────────────────────────
results_summary = pd.DataFrame({
    'Model': ['Multiple Linear Regression', 'Decision Tree', 'Random Forest',
              'Gradient Boosting', 'KNN'],
    'R2':   [mlr_r2,  dtr_r2,  rfr_r2,  gb_r2,  knn_r2],
    'MAE':  [mlr_mae, dtr_mae, rfr_mae, gb_mae, knn_mae],
    'RMSE': [mlr_rmse,dtr_rmse,rfr_rmse,gb_rmse,knn_rmse],
}).sort_values('R2', ascending=False).reset_index(drop=True)
print(results_summary.round(4).to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, metric in zip(axes, ['R2', 'MAE', 'RMSE']):
    better_low = metric in ('MAE', 'RMSE')
    vals = results_summary.set_index('Model')[metric]
    colors = ['#2e8b57' if v == (vals.min() if better_low else vals.max()) else '#5588cc'
              for v in vals]
    ax.barh(vals.index, vals.values, color=colors)
    for i, v in enumerate(vals.values): ax.text(v + 0.005, i, f'{v:.4f}', va='center', fontsize=9)
    ax.set_title(f'{metric} ({"lower" if better_low else "higher"} is better)', fontweight='bold')
    ax.set_xlabel(metric)
plt.suptitle('Model Comparison: Solar Irradiance Prediction, Mustang Nepal', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()